# UFC Data Cleaning\n\nProcess raw scraped data into analysis-ready DataFrames.\n\n- **Fighter 1** = Red corner = Favorite\n- **Fighter 2** = Blue corner = Underdog\n- Pre-2015 data dropped (less relevant to modern UFC)

In [ ]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import re

DATA_DIR = "./data"
print("✅ Ready")

In [ ]:
# Cell 2 — Load Raw Data
events = pd.read_csv(f"{DATA_DIR}/events.csv")
fights = pd.read_csv(f"{DATA_DIR}/fights.csv")
fighters = pd.read_csv(f"{DATA_DIR}/fighters.csv")

with open(f"{DATA_DIR}/fight_details.json", "r") as f:
    fight_details = json.load(f)

print(f"Events:        {events.shape}")
print(f"Fights:        {fights.shape}")
print(f"Fighters:      {fighters.shape}")
print(f"Fight details: {len(fight_details)}")

f1_wr = (fights["winner"] == fights["fighter_1"]).mean()
print(f"\nRed corner win rate: {f1_wr:.1%}")

In [ ]:
# Cell 3 — Data Quality by Year
events["event_date_parsed"] = pd.to_datetime(events["event_date"], format="mixed", errors="coerce")
event_date_map = events.set_index("event_name")["event_date_parsed"]
fights["event_date_parsed"] = fights["event_name"].map(event_date_map)
fights["year"] = fights["event_date_parsed"].dt.year

print("FIGHTS PER YEAR:")
yearly_counts = fights["year"].value_counts().sort_index()
print(yearly_counts.to_string())

print("\nMISSING DATA % BY YEAR:")
yearly_missing = fights.groupby("year").apply(
    lambda x: x[["f1_str", "f2_str", "f1_td", "f2_td"]].isna().mean().mean() * 100
).round(1)
print(yearly_missing.to_string())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
yearly_counts.plot(kind="bar", ax=axes[0])
axes[0].set_title("Fights Per Year", fontweight="bold")
axes[0].set_ylabel("Count")

yearly_missing.plot(kind="bar", ax=axes[1], color="orange")
axes[1].set_title("Avg Missing Data % By Year", fontweight="bold")
axes[1].set_ylabel("Missing %")

plt.tight_layout()
plt.show()

total = len(fights)
post_2015 = len(fights[fights["year"] >= 2015])
print(f"\nTotal fights: {total}")
print(f"Post-2015 fights: {post_2015} ({post_2015/total:.1%})")
print(f"Would drop: {total - post_2015} ({(total-post_2015)/total:.1%})")

In [ ]:
# Cell 4 — Apply Year Cutoff
CUTOFF_YEAR = 2015

fights = fights[fights["year"] >= CUTOFF_YEAR].reset_index(drop=True)
print(f"✅ Filtered to {CUTOFF_YEAR}+ → {len(fights)} fights remaining")
f1_wr = (fights["winner"] == fights["fighter_1"]).mean()
print(f"Red corner win rate (post-{CUTOFF_YEAR}): {f1_wr:.1%}")

In [ ]:
# Cell 5 — Clean Events
events_clean = events.copy()
events_clean["event_date"] = pd.to_datetime(events_clean["event_date"], format="mixed", errors="coerce")
events_clean = events_clean.sort_values("event_date", ascending=False).reset_index(drop=True)
events_clean = events_clean[events_clean["event_date"].dt.year >= CUTOFF_YEAR].reset_index(drop=True)

print(f"Events after {CUTOFF_YEAR} cutoff: {len(events_clean)}")
print(f"Date range: {events_clean['event_date'].min()} to {events_clean['event_date'].max()}")
events_clean.head()

In [ ]:
# Cell 6 — Clean Fighters
def parse_height_to_inches(height_str):
    if pd.isna(height_str) or str(height_str).strip() in ("", "--"):
        return np.nan
    match = re.search(r"(\d+)'\s*(\d+)\"?", str(height_str))
    if match:
        feet, inches = int(match.group(1)), int(match.group(2))
        return feet * 12 + inches
    return np.nan

def parse_reach_to_inches(reach_str):
    if pd.isna(reach_str) or str(reach_str).strip() in ("", "--"):
        return np.nan
    match = re.search(r"([\d.]+)", str(reach_str))
    return float(match.group(1)) if match else np.nan

def parse_weight_to_lbs(weight_str):
    if pd.isna(weight_str) or str(weight_str).strip() in ("", "--"):
        return np.nan
    match = re.search(r"([\d.]+)", str(weight_str))
    return float(match.group(1)) if match else np.nan

fighters_clean = fighters.copy()
fighters_clean["full_name"] = (fighters_clean["first_name"].fillna("") + " " + fighters_clean["last_name"].fillna("")).str.strip()
fighters_clean["height_inches"] = fighters_clean["height"].apply(parse_height_to_inches)
fighters_clean["reach_inches"] = fighters_clean["reach"].apply(parse_reach_to_inches)
fighters_clean["weight_lbs"] = fighters_clean["weight"].apply(parse_weight_to_lbs)

for col in ["wins", "losses", "draws"]:
    fighters_clean[col] = pd.to_numeric(fighters_clean[col], errors="coerce").fillna(0).astype(int)

fighters_clean["total_fights"] = fighters_clean["wins"] + fighters_clean["losses"] + fighters_clean["draws"]
fighters_clean["win_pct"] = np.where(fighters_clean["total_fights"] > 0, fighters_clean["wins"] / fighters_clean["total_fights"], 0)

print(f"Fighters: {fighters_clean.shape}")
fighters_clean.head()

In [ ]:
# Cell 7 — Clean Fights
def parse_strikes(strike_str):
    if pd.isna(strike_str) or str(strike_str).strip() in ("", "--"):
        return np.nan, np.nan
    match = re.search(r"(\d+)\s+of\s+(\d+)", str(strike_str))
    if match:
        return int(match.group(1)), int(match.group(2))
    return np.nan, np.nan

fights_clean = fights.copy()

# Map event dates
event_dates = events_clean.set_index("event_name")["event_date"]
fights_clean["event_date"] = fights_clean["event_name"].map(event_dates)
fights_clean["event_date"] = pd.to_datetime(fights_clean["event_date"], format="mixed", errors="coerce")

# Parse strikes and takedowns for both corners
for prefix in ["f1", "f2"]:
    landed, attempted = zip(*fights_clean[f"{prefix}_str"].apply(parse_strikes))
    fights_clean[f"{prefix}_str_landed"] = pd.Series(landed)
    fights_clean[f"{prefix}_str_attempted"] = pd.Series(attempted)
    fights_clean[f"{prefix}_str_acc"] = np.where(
        fights_clean[f"{prefix}_str_attempted"] > 0,
        fights_clean[f"{prefix}_str_landed"] / fights_clean[f"{prefix}_str_attempted"],
        0
    )

    td_landed, td_attempted = zip(*fights_clean[f"{prefix}_td"].apply(parse_strikes))
    fights_clean[f"{prefix}_td_landed"] = pd.Series(td_landed)
    fights_clean[f"{prefix}_td_attempted"] = pd.Series(td_attempted)

    fights_clean[f"{prefix}_kd"] = pd.to_numeric(fights_clean[f"{prefix}_kd"], errors="coerce")
    fights_clean[f"{prefix}_sub"] = pd.to_numeric(fights_clean[f"{prefix}_sub"], errors="coerce")

# Parse round and time
fights_clean["round"] = pd.to_numeric(fights_clean["round"], errors="coerce")

def time_to_seconds(t):
    if pd.isna(t) or str(t).strip() in ("", "--"):
        return np.nan
    match = re.match(r"(\d+):(\d+)", str(t))
    if match:
        return int(match.group(1)) * 60 + int(match.group(2))
    return np.nan

fights_clean["time_seconds"] = fights_clean["time"].apply(time_to_seconds)
fights_clean["total_time_seconds"] = ((fights_clean["round"] - 1) * 5 * 60) + fights_clean["time_seconds"]

# Target variable: did red corner (fighter_1 / favorite) win?
fights_clean["f1_win"] = (fights_clean["winner"] == fights_clean["fighter_1"]).astype(int)

# Clean method
fights_clean["method_clean"] = fights_clean["method"].str.strip().str.upper()
fights_clean["finish_type"] = fights_clean["method_clean"].apply(lambda x:
    "KO/TKO" if "KO" in str(x) else
    "SUB" if "SUB" in str(x) else
    "DEC" if "DEC" in str(x) else
    "OTHER"
)

fights_clean["weight_class"] = fights_clean["weight_class"].str.strip()

# Drop temp columns
fights_clean = fights_clean.drop(columns=["event_date_parsed", "year"], errors="ignore")

print(f"Fights: {fights_clean.shape}")
print(f"\n🔴 Red corner (F1/favorite) win rate: {fights_clean['f1_win'].mean():.1%}")
print(f"🔵 Blue corner (F2/underdog) win rate: {1 - fights_clean['f1_win'].mean():.1%}")
fights_clean.head()

In [ ]:
# Cell 8 — Save Cleaned Data
events_clean.to_csv(f"{DATA_DIR}/events_clean.csv", index=False)
fighters_clean.to_csv(f"{DATA_DIR}/fighters_clean.csv", index=False)
fights_clean.to_csv(f"{DATA_DIR}/fights_clean.csv", index=False)

print("✅ Cleaned data saved:")
print(f"   Events:   {len(events_clean)} rows → {DATA_DIR}/events_clean.csv")
print(f"   Fighters: {len(fighters_clean)} rows → {DATA_DIR}/fighters_clean.csv")
print(f"   Fights:   {len(fights_clean)} rows → {DATA_DIR}/fights_clean.csv")
print(f"\n   Year cutoff: {CUTOFF_YEAR}+")
print(f"   🔴 Red corner (favorite) win rate: {fights_clean['f1_win'].mean():.1%} ← baseline to beat")